**Exemple 1: Quantifier l’implicite lors des sessions Questions/ Réponses entre analystes financiers et dirigeants**

In [ ]:
import pandas as pd
from pydantic import BaseModel, Field
import json
from openai import OpenAI
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

**Préparation de la donnée**

Création faux Dataset

In [ ]:
data = {"entreprise_id": [1, 2, 3, 4, 5, 6, 7, 8],
        "reponse_pdg": [
        "Nos résultats sont excellents, dette réduite.",
        "Il est encore trop tôt pour donner un chiffre, plusieurs pistes sont à l'étude.",
        "Ne nous focalisons pas sur ces difficultés.",
        "Ratio sous contrôle, plan validé.",
        "Tout est excellent, la trésorerie est bonne.",
        "Difficile à dire, c'est trop tôt pour s'engager.",
        "Ces difficultés sont passagères, regardons l'avenir.",
        "Le bilan est clair, l'EBITDA est sous contrôle."
        ],
        "ratio_liquidite": [1.5, 0.8, 0.9, 1.4, 1.6, 0.7, 0.8, 1.3],
        "risque_baisse": [0, 1, 1, 0, 0, 1, 1, 0]
}
# Transformation de dictionnaire en tableau Pandas
df = pd.DataFrame(data)

**Interaction avec le LLM**

In [ ]:
# Configuration OpenAI au cas où vraie clé API
client = OpenAI(api_key="CLE_API")

# Classe qui hérite de BaseModel qui est une fonctionnalité de Pydantic
# Pour obliger API de OpenAI de répondre avec du json
class AnalyseComportementale(BaseModel):
    score_esquive: int = Field(description="Note sur 10 de l'esquive")
    score_defensif: int = Field(description="Note sur 10 du ton défensif")

# Fonction qui remplace la vraie IA
def analyser_texte_llm(texte):
    texte = texte.lower() #tout en minuscules
    # Si le PDG est confiant
    if "excellents" in texte or "sous contrôle" in texte:
        return 1, 1
    # Note d'esquive
    elif "trop tôt" in texte or "plusieurs pistes" in texte:
        return 9, 7
    # Note défensive
    elif "focaliser" in texte or "difficultés" in texte:
        return 5, 9
    return 5, 5 # Ce résultat si aucun de tout cela

# Préparation des listes vides pour stocker futures notes
liste_esquive = []
liste_defensif = []

# Parcourt chaque ligne du tableau
for texte in df['reponse_pdg']:
    note_esquive, note_defensif = analyser_texte_llm(texte) # Attribution des scores
    # Ajout des notes obtenues dans les listes
    liste_esquive.append(note_esquive)
    liste_defensif.append(note_defensif)

# Deux nouvelles colonnes dans le tableau
df['feature_esquive'] = liste_esquive
df['feature_defensif'] = liste_defensif

print("Le  tableau après analyse :")
print(df[['reponse_pdg', 'feature_esquive', 'feature_defensif']])

Le  tableau après analyse :
                                         reponse_pdg  feature_esquive  \
0      Nos résultats sont excellents, dette réduite.                1   
1  Il est encore trop tôt pour donner un chiffre,...                9   
2        Ne nous focalisons pas sur ces difficultés.                5   
3                  Ratio sous contrôle, plan validé.                1   
4       Tout est excellent, la trésorerie est bonne.                5   
5   Difficile à dire, c'est trop tôt pour s'engager.                9   
6  Ces difficultés sont passagères, regardons l'a...                5   
7    Le bilan est clair, l'EBITDA est sous contrôle.                1   

   feature_defensif  
0                 1  
1                 7  
2                 9  
3                 1  
4                 5  
5                 7  
6                 9  
7                 1  


**Integration Machine Learning**

In [ ]:
# Tableau des features
X = df[['ratio_liquidite', 'feature_esquive', 'feature_defensif']]

# Variable cible
y = df['risque_baisse']

# On garde 25% des données pour vérifier l'algo à la fin
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=12)

# On choisit l'algorithme Random Forest
# Création de 100 arbres de décision puis garder meilleure solution
modele = RandomForestClassifier(n_estimators=100, random_state=42)
modele.fit(X_train, y_train) # Entrainement du modele

# Predictions sur les 25% restants
predictions = modele.predict(X_test)
precision = accuracy_score(y_test, predictions) # Comparaison des predictions
print(f"Précision : {precision * 100}%")

print("\nCe qui a le plus compté pour l'algorithme :")
for nom_colonne, importance in zip(X.columns, modele.feature_importances_):
    print(f" - {nom_colonne} : {importance:.2f}")

Précision : 100.0%

Ce qui a le plus compté pour l'algorithme :
 - ratio_liquidite : 0.36
 - feature_esquive : 0.35
 - feature_defensif : 0.29
